In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

source_folder = '/content/drive/MyDrive/Colab Notebooks/Hackathon # Trial_3/File_Table/'
import os

print(f"Checking folder: {source_folder}")
print("Contents of '/content/drive/MyDrive/':")
!ls -F '/content/drive/MyDrive/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking folder: /content/drive/MyDrive/Colab Notebooks/Hackathon # Trial_3/File_Table/
Contents of '/content/drive/MyDrive/':
 18289.pdf
 AI/
'Colab Notebooks'/
 ColabNotebooks/
 F6628390-EFC1-4B17-BD51-9A7189D72B03.jpeg
'✅Free SILENT LETTERS.pdf'
 Github/
'How to get started with Drive.pdf'
'IG login.mp4'
 IMG_2081.MOV
'India survey'/
'Japan Itinerary_Revised 6.xlsx'
 path_to_your_json_results/
'Resume_Suthinee Nak-udom.pdf'
 Revised_タイの仏教.pptx
'Summary Built-in (1).pptx'
'Summary Built-in.pptx'
'Untitled map.gmap'
'xx N2 Grammar FlashCard (Full) xx.pdf'
 タイの宝くじ.pptx
 タイの結婚式.pptx
 履歴書＿スティニー・ナックウドム.pdf
'現場でよく使う単語 (ชุดที่1)-1.pdf'
'現場でよく使う単語 (ชุดที่2).pdf'
 職務経歴＿スティニー・ナックウドム.pdf


In [ ]:
import os
import shutil
import cv2

def detect_table_final_filter(image_path):
    img = cv2.imread(image_path)
    # ถ้าอ่านรูปไม่ได้ ส่งค่า 0 กลับไปแทน False
    if img is None: return 0

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # ทำภาพขาวดำ
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 5)

    # ค้นหาเส้นนอนและเส้นตั้งที่ยาวพอจะเป็นตาราง
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 50))

    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel, iterations=2)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel, iterations=2)

    # รวมเส้นสร้างเป็นตาราง
    table_structure = cv2.add(h_lines, v_lines)

    # ค้นหาช่องว่างในตาราง
    contours, _ = cv2.findContours(table_structure, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # นับจำนวนช่องที่มีขนาดเหมาะสม
    table_cells = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h
        # กรองเอาเฉพาะช่องที่มีขนาดเหมือน ช่องตาราง
        if 500 < area < (img.shape[0] * img.shape[1] * 0.5):
            table_cells.append(cnt)

    # ส่งค่าเป็น จำนวนช่อง ออกมาแทน เพื่อให้เอาไปใช้ต่อได้
    return len(table_cells)

# --- ส่วนการรัน ---
print(f"🚀 กำลังคัดแยกด้วยระบบ Cell Density Check...")

# บังคับสร้างโฟลเดอร์ปลายทาง ป้องกัน Error: FileNotFoundError
os.makedirs(target_folder, exist_ok=True)

count_moved = 0
for file_name in files:
    img_path = os.path.join(source_folder, file_name)
    destination_path = os.path.join(target_folder, file_name)

    if not os.path.exists(img_path): continue

    # ดึงจำนวนช่องมาเก็บในตัวแปร cell_count
    cell_count = detect_table_final_filter(img_path)

    # ถ้าจำนวนช่องมีมากกว่า 8 ถือว่าเป็นตาราง
    if cell_count > 8:
        if not os.path.exists(destination_path):
            shutil.copy(img_path, destination_path)

            print(f"✅ {file_name}: ตรวจพบตาราง (มี {cell_count} ช่อง)")
            count_moved += 1
    else:
        # ลบไฟล์ที่หลุดมาคราวที่แล้วออก
        if os.path.exists(destination_path):
            os.remove(destination_path)
            print(f"🗑️ {file_name}: คัดออก (มีแค่ {cell_count} ช่อง ไม่ใช่ตารางข้อมูล)")

print(f"✨ เสร็จสมบูรณ์! คัดแยกได้แม่นยำขึ้น {count_moved} ไฟล์")

In [ ]:
!pip install azure-ai-formrecognizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.4/301.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os
import json
import time
import re
from natsort import natsorted
from IPython.display import display

# --- นำเข้า Library ของ Azure ---
from azure.core.credentials import AzureKeyCredential
from azure.ai.formrecognizer import DocumentAnalysisClient

# --- ตั้งค่า Path ---
TABLE_DIR = '/content/drive/MyDrive/Colab Notebooks/Hackathon # Trial_3/File_Table/'
CACHE_DIR = '/content/drive/MyDrive/Colab Notebooks/Hackathon #2 Trial_3/ocr_json_results_azure/'

os.makedirs(CACHE_DIR, exist_ok=True)

# 🔑 ใส่ Key และ Endpoint ของ Azure ตรงนี้
AZURE_ENDPOINT = "https://hackathon2.cognitiveservices.azure.com/".strip()
AZURE_KEY = " YOUR API KEY".strip()

# ตั้งค่า Client
document_client = DocumentAnalysisClient(
    endpoint=AZURE_ENDPOINT, credential=AzureKeyCredential(AZURE_KEY)
)

# --- [ฟังก์ชันช่วยทำความสะอาดตัวเลข] ---
def thai_to_arabic(text):
    if not isinstance(text, str): return text
    trans = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    return text.translate(trans)

def clean_vote_text(text):
    if pd.isna(text) or str(text).strip() == '': return None
    text_str = str(text).split('(')[0].strip()
    numbers_only = re.sub(r'[^0-9]', '', thai_to_arabic(text_str))
    return int(numbers_only) if numbers_only else None

# --- ฟังก์ชันดึงข้อมูลด้วย Azure Document Intelligence ---
def call_azure_and_extract_tables(image_path, cache_file):
    print(f"📡 กำลังส่ง {os.path.basename(image_path)} ไปหา Azure...")
    try:
        with open(image_path, "rb") as f:
            poller = document_client.begin_analyze_document("prebuilt-layout", document=f)

        result = poller.result()
        all_tables_grid = []

        # วนลูปอ่านทุกตารางที่ Azure เจอ
        for table in result.tables:
            # สร้างตาราง 2 มิติ (List of Lists) ว่างๆ มารองรับ
            table_grid = [["" for _ in range(table.column_count)] for _ in range(table.row_count)]

            # หยอดข้อมูลลงตามพิกัด Row/Column
            for cell in table.cells:
                table_grid[cell.row_index][cell.column_index] = cell.content

            all_tables_grid.append(table_grid)

        # เซฟลง Cache เป็น JSON
        if all_tables_grid:
            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump({"tables": all_tables_grid}, f, ensure_ascii=False, indent=4)
            print(f"💾 เซฟตารางสำเร็จ! หน่วงเวลา 3 วิ...")
            time.sleep(3) # Azure Tier ฟรี (F0) รับได้ประมาณ 20 หน้า/นาที หน่วงไว้ 3 วิปลอดภัยครับ
            return all_tables_grid
        return None

    except Exception as e:
        print(f"⚠️ เกิดข้อผิดพลาดกับ Azure: {e}")
        return None

# --- ฟังก์ชันแยกพรรคและคะแนนจาก Grid ---
def parse_azure_grid_to_dict(all_tables_grid, source_filename):
    data_list = []
    try:
        for grid in all_tables_grid:
            if not grid or len(grid) < 2: continue

            # จำนวนคอลัมน์ในตาราง
            num_cols = len(grid[0])
            p_idx = 2 if num_cols > 2 else 1
            v_idx = 3 if num_cols > 3 else num_cols - 1

            for row in grid:
                try:
                    p_name = str(row[p_idx]).strip()
                    # ข้ามพวกหัวตาราง
                    if any(w in p_name for w in ['พรรค', 'สังกัด', 'รวม', 'nan', 'หมายเหตุ']): continue
                    if p_name == "": continue # ข้ามบรรทัดว่าง

                    val = clean_vote_text(row[v_idx])
                    if val is not None:
                        data_list.append({
                            "source_file": source_filename,
                            "party_name": p_name,
                            "votes": val
                        })
                except: continue
        return data_list
    except: return []

# ==========================================
# 🛑 ส่วนการรัน Loop ในโฟลเดอร์ File_Table
# ==========================================
final_results = []

if os.path.exists(TABLE_DIR):
    all_files = [f for f in os.listdir(TABLE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    files_to_process = natsorted(all_files)
    print(f"🔎 พบไฟล์ใน File_Table ทั้งหมด {len(files_to_process)} ไฟล์\n")

    for file_name in files_to_process:
        img_path = os.path.join(TABLE_DIR, file_name)
        cache_file = os.path.join(CACHE_DIR, f"{os.path.splitext(file_name)[0]}.json")

        tables_grid = None

        # ระบบ Cache: ถ้าทำไปแล้วไม่ต้องส่ง Azure ซ้ำ
        if os.path.exists(cache_file):
            print(f"⏩ Resume (โหลด Cache): {file_name}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                tables_grid = json.load(f).get("tables", [])
        else:
            tables_grid = call_azure_and_extract_tables(img_path, cache_file)

        if tables_grid:
            extracted = parse_azure_grid_to_dict(tables_grid, file_name)
            final_results.extend(extracted)
            if extracted: print(f"✅ สำเร็จ: สกัดได้ {len(extracted)} พรรค")
        print("-" * 30)
else:
    print(f"❌ ไม่พบโฟลเดอร์: {TABLE_DIR}")

# --- ขั้นตอนประกอบร่างเป็น CSV ---
if final_results:
    df_raw = pd.DataFrame(final_results)

    # ตัด _page ออกเพื่อทำ doc_id
    df_raw['doc_id'] = df_raw['source_file'].apply(lambda x: re.sub(r'_page\d+', '', os.path.splitext(x)[0], flags=re.IGNORECASE))

    # รันตัวเลขต่อเนื่อง
    df_raw['row_num'] = df_raw.groupby('doc_id').cumcount() + 1
    df_raw['id'] = df_raw['doc_id'] + '_' + df_raw['row_num'].astype(str)

    # จัดคอลัมน์
    df_final = df_raw[['id', 'doc_id', 'row_num', 'party_name', 'votes']]

    # บันทึกไฟล์
    df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

    print(f"\n✨ เสร็จสิ้น! รวมข้อมูลทั้งหมดด้วย Azure ได้ {len(df_final)} แถว")
    print(f"📁 บันทึกไปที่: {OUTPUT_CSV}")
    display(df_final.head(15))

🔎 พบไฟล์ใน File_Table ทั้งหมด 632 ไฟล์

⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_1_page2.png
✅ สำเร็จ: สกัดได้ 17 พรรค
------------------------------
⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_2_page2.png
✅ สำเร็จ: สกัดได้ 14 พรรค
------------------------------
⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_3_page2.png
✅ สำเร็จ: สกัดได้ 13 พรรค
------------------------------
⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_4.png
✅ สำเร็จ: สกัดได้ 2 พรรค
------------------------------
⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_4_page2.png
✅ สำเร็จ: สกัดได้ 10 พรรค
------------------------------
⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_5_page2.png
------------------------------
⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_6_page2.png
✅ สำเร็จ: สกัดได้ 15 พรรค
------------------------------
⏩ Resume (โหลด Cache): Copy of Copy of constituency_10_7_page2.png
✅ สำเร็จ: สกัดได้ 12 พรรค
------------------------------
⏩

,id,doc_id,row_num,party_name,votes
0,Copy of Copy of constituency_10_1_1,Copy of Copy of constituency_10_1,1,ประชาชน,34167
1,Copy of Copy of constituency_10_1_2,Copy of Copy of constituency_10_1,2,ประชาธิปัตย์,14813
2,Copy of Copy of constituency_10_1_3,Copy of Copy of constituency_10_1,3,ภูมิใจไทย,14368
3,Copy of Copy of constituency_10_1_4,Copy of Copy of constituency_10_1,4,เพื่อไทย,6030
4,Copy of Copy of constituency_10_1_5,Copy of Copy of constituency_10_1,5,โอกาสใหม่,1133
5,Copy of Copy of constituency_10_1_6,Copy of Copy of constituency_10_1,6,ไทยภักดี,1023
6,Copy of Copy of constituency_10_1_7,Copy of Copy of constituency_10_1,7,เศรษฐกิจ,979
7,Copy of Copy of constituency_10_1_8,Copy of Copy of constituency_10_1,8,ไทยสร้างไทย,629
8,Copy of Copy of constituency_10_1_9,Copy of Copy of constituency_10_1,9,ไทยก้าวใหม่,489
9,Copy of Copy of constituency_10_1_10,Copy of Copy of constituency_10_1,10,พลวัต,351


In [ ]:
import os
import json
import pandas as pd
import re
from collections import defaultdict

# --- แปลงเลขไทย -> อารบิก ---
def clean_to_int(text):
    if text is None or str(text).lower() == 'nan' or str(text).strip() == '': return 0
    text_fixed = str(text).replace('.', ',')
    thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    txt = text_fixed.translate(thai_to_arabic)
    nums = re.findall(r'\d+', txt.replace(',', ''))
    return int(nums[0]) if nums else 0

# --- ล้างชื่อพรรค ---
def clean_party(name):
    # เพิ่มกันเหนียว เผื่อ Excel ช่องนั้นเป็นค่าว่าง
    if pd.isna(name) or not name: return ""
    t = str(name).replace("พรรค", "").replace('"', '').replace(',', '')
    t = t.replace(":selected:", "").replace(":unselected:", "")
    return t.strip()

# --- ดึงข้อมูล ---
def get_sequential_data(data, doc_id):
    results = []
    if 'tables' in data:
        for table in data['tables']:
            for row in table:
                clean_r = [str(x).strip() for x in row if str(x).strip() != ""]

                if len(clean_r) >= 3 and "หมายเลข" not in clean_r[0] and "คะแนน" not in clean_r[-1] and "พรรค" not in clean_r[0]:

                    # เช็คว่าเป็นตัวเลขไหม (กันตารางสรุปยอด)
                    first_col = clean_r[0].translate(str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789'))
                    if not re.search(r'\d', first_col):
                        continue

                    if 'party_list' in str(doc_id).lower():
                        p_name = clean_party(clean_r[1])
                        v_count = clean_to_int(clean_r[2])
                    else:
                        if len(clean_r) >= 4:
                            p_name = clean_party(clean_r[2])
                            v_count = clean_to_int(clean_r[3])
                        else:
                            p_name = clean_party(clean_r[-2])
                            v_count = clean_to_int(clean_r[-1])

                    if "คน" in p_name or "บัตร" in p_name or "สิทธิ" in p_name:
                        continue

                    if p_name or v_count > 0:
                        results.append({"party": p_name, "vote": v_count})
    return results

# --- หาหน้า ---
def get_page_number(filename):
    if "_page" not in filename: return 0
    match = re.search(r'_page(\d+)', filename)
    return int(match.group(1)) if match else 999

# --- ตั้งค่า Path ---
json_dir = "/content/drive/MyDrive/Colab Notebooks/Hackathon #2 Trial_3/ocr_json_results_azure"
csv_path = "/content/drive/MyDrive/Colab Notebooks/Hackathon #2 Trial_3/submission_template.csv"
output_path = "/content/drive/MyDrive/Colab Notebooks/Hackathon #2 Trial_3/603364_Suthinee.csv"

df = pd.read_csv(csv_path)

# บังคับแปลง ID ใน Excel ให้เป็นตัวพิมพ์เล็กทั้งหมด!
df['match_id'] = df['doc_id'].astype(str).str.strip().str.lower()

json_groups = defaultdict(list)
if os.path.exists(json_dir):
    for f in os.listdir(json_dir):
        if f.endswith(".json"):
            clean_name = re.sub(r'(?i)(copy\s+of\s+)+', '', f).strip()
            base_id = re.sub(r'_page\d+.*', '', clean_name).replace(".json", "").strip()
            json_groups[base_id].append(f)

print(f"🚀 เริ่มดึงข้อมูล (โหมด Match คะแนนตามชื่อพรรค Base on โค้ดของกิ๊ฟ!) ...")

for bid, files in json_groups.items():
    sorted_files = sorted(files, key=get_page_number)

    all_data_for_id = []
    for f_name in sorted_files:
        with open(os.path.join(json_dir, f_name), 'r', encoding='utf-8') as f:
            try:
                content = json.load(f)
                all_data_for_id.extend(get_sequential_data(content, bid))
            except: continue

    # สร้าง Dictionary จับคู่ "ชื่อพรรค: คะแนน" เพื่อเตรียมไว้ Match
    vote_map = {item['party']: item['vote'] for item in all_data_for_id}

    # [จุดแก้ปัญหา 2] บังคับแปลง ID จากไฟล์ JSON เป็นพิมพ์เล็กก่อนจับคู่!
    mask = df['match_id'] == bid.lower()
    target_indices = df[mask].index

    if len(target_indices) > 0:
        for idx in target_indices:
            # อ่านชื่อพรรคจาก Excel ออกมาทำความสะอาด
            csv_party = clean_party(df.at[idx, 'party_name'])

            # หยอดเฉพาะคะแนนลงไป โดยค้นหาจาก Dictionary (ไม่แตะชื่อพรรคเดิม)
            df.at[idx, 'votes'] = vote_map.get(csv_party, 0)

# ลบคอลัมน์ตัวช่วยทิ้งไปก่อนเซฟ จะได้เนียนเหมือนเดิม
df.drop(columns=['match_id'], inplace=True)

df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✨ เสร็จเรียบร้อย! โหลดไฟล์ {output_path} เช็คได้เลยครับ ฟอร์แมตเดิมเป๊ะ 100%!")

# โชว์ผลลัพธ์ของ constituency_14_2 ให้ดูว่ามาแล้วจริงๆ
display(df[df['doc_id'] == 'constituency_14_2'][['doc_id', 'party_name', 'votes']].head(5))